# RAG Pipelines- Data Ingestion to Vector DB Pipeline

In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\NAVEEN\AppData\Local\Temp\ipykernel_14356\3933654057.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader


## RAG Creation

### Read all the pdf's inside directory

In [8]:
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = [ ]
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:

            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file' ] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            all_documents.extend(documents)
            print(f" Loaded {len(documents)} pages")
        except Exception as e:
            print(f" X Error: {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 2 PDF files to process

Processing: attention.pdf
 Loaded 15 pages

Processing: Parallel_Attention.pdf
 Loaded 10 pages

Total documents loaded: 25


### Text splitting get into chunks

In [9]:
def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    "Split documents into smaller chunks for better RAG performance"
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ",  ""]
    )
    
    split_docs = text_splitter.split_documents(documents)  # passing the list of all documents
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

In [10]:
chunks = split_documents(all_pdf_documents)

Split 25 documents into 113 chunks

Example chunk:
Content: Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz...
Metadata: {'producer': 'pdfTeX-1.40.17', 'creator': 'LaTeX with hyperref package', 'creationdate': '2017-06-13T00:21:17+00:00', 'author': '', 'keywords': '', 'moddate': '2017-06-13T00:21:17+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.17 (TeX Live 2016) kpathsea version 6.2.2', 'subject': '', 'title': '', 'trapped': '/False', 'source': '..\\data\\pdf\\attention.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1', 'source_file': 'attention.pdf', 'file_type': 'pdf'}


### embedding And vectorStoreDB

In [11]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [12]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def  __init__(self, model_name: str = "all-MiniLM-L6-v2"):  # this model is avaiable on hugging face
        """
            Initialize the embedding manager
            Args :
                model_name: HuggingFace model name for sentence embeddings
        """

        self.model_name = model_name
        self.model = None
        self._load_model()
        
    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}") ## somewhere around 384
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise
    
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts

        Args:
            texts: List of text strings to embed

        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)

        """
        if not self.model:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings. shape}")
        return embeddings
    

In [13]:
## initialize the embedding manager

embedding_manager=EmbeddingManager() 

Loading embedding model: all-MiniLM-L6-v2


d:\Languages\python\LangTool\RAG\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\NAVEEN\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
d:\Languages\python\LangTool\RAG\.venv\Lib\site-packages\huggingface_hub\file_download.py:

Model loaded successfully. Embedding dimension: 384


### Vector store

In [19]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store

        Args :
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()
    
    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try: 
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection. count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise
        
    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store

        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """

        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        print(f"Adding {len(documents)} documents to vector store ...")
        

        # Prepare data for ChromaDB
        ids = []
        metadatas = [ ]
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append (metadata)
            
            # Document qontent
            documents_text.append(doc.page_content)

            # Embedding
            embeddings_list.append(embedding.tolist())
            
        # Add to collection
        try:
            self.collection. add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

In [20]:
vectorstore = VectorStore()

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


### Convert the text to embeddings


In [23]:
texts=[doc.page_content for doc in chunks]  # combine only all texts for embedding
# texts

In [21]:
## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks, embeddings)

Generating embeddings for 113 texts...


Batches: 100%|██████████| 4/4 [00:03<00:00,  1.23it/s]

Generated embeddings with shape: (113, 384)
Adding 113 documents to vector store ...
Successfully added 113 documents to vector store
Total documents in collection: 113


### Retriever Pipeline From VectorStore

In [24]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever

        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query

        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold

        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query:  '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings ([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            # Process results
            retrieved_docs = []

            if results['documents'] and results['documents' ] [0] :
                documents = results['documents'] [0]
                metadatas = results['metadatas'] [0]
                distances = results['distances'] [0]
                ids = results['ids' ] [0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs. append ({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
                
            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

### Query Test

In [25]:
rag_retriever=RAGRetriever(vectorstore, embedding_manager)

In [27]:
rag_retriever.retrieve("What is Encoder and Decoder Stacks")

Retrieving documents for query:  'What is Encoder and Decoder Stacks'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 69.39it/s]

Generated embeddings with shape: (1, 384)
Retrieved 4 documents (after filtering)


[{'id': 'doc_ab74a5de_8',
  'content': 'Encoder: The encoder is composed of a stack of N = 6 identical layers. Each layer has two\nsub-layers. The ﬁrst is a multi-head self-attention mechanism, and the second is a simple, position-\nwise fully connected feed-forward network. We employ a residual connection around each of\nthe two sub-layers, followed by layer normalization [ 1]. That is, the output of each sub-layer is\nLayerNorm(x + Sublayer(x)), where Sublayer(x) is the function implemented by the sub-layer\nitself. To facilitate these residual connections, all sub-layers in the model, as well as the embedding\nlayers, produce outputs of dimensiondmodel = 512.\nDecoder: The decoder is also composed of a stack ofN = 6 identical layers. In addition to the two\nsub-layers in each encoder layer, the decoder inserts a third sub-layer, which performs multi-head\nattention over the output of the encoder stack. Similar to the encoder, we employ residual connections\n2',
  'metadata': {'page_

## Integration Vectordb Context pipeline With LLM output

In [30]:
from langchain_groq import ChatGroq

import os 
from dotenv import load_dotenv
load_dotenv()

True

In [32]:
groq_api_key = os.getenv("GROQ_API_KEY")

llm = ChatGroq(
        api_key=groq_api_key,
        model='qwen/qwen3.8-27b', 
        temperature=0.1, 
        max_tokens=1024
    )

#### Simple RAG function: retrieve context + generate response


In [33]:
def rag_simple(query,retriever,llm,top_k=3):
    
    ## retriever the context
    results=retriever.retrieve(query, top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""

    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:
        """
    response=llm.invoke([prompt.format(context=context, query=query) ])
    return response.content

In [34]:
answer=rag_simple("What is Encoder and Decoder Stacks?", rag_retriever, llm)
print(answer)

Retrieving documents for query:  'What is Encoder and Decoder Stacks?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 60.79it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


The Encoder and Decoder Stacks are the two main components of the Transformer model, each composed of a stack of N = 6 identical layers.

*   **Encoder:** Each layer contains two sub-layers: a multi-head self-attention mechanism and a position-wise fully connected feed-forward network.
*   **Decoder:** Each layer contains three sub-layers: a masked multi-head self-attention mechanism, a multi-head attention mechanism over the encoder output, and a position-wise fully connected feed-forward network.

Both stacks utilize residual connections and layer normalization around each sub-layer, with all sub-layers producing outputs of dimension $d_{model} = 512$.


#### Enhanced RAG Pipeline Features

In [35]:
# --- Enhanced RAG Pipeline Features
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """

    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}

    # Prepare context and sources
    context = "\n\n". join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown' )),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:120] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely. \nContext: \n{context} \n\nQuestion{query} \n\nAnswer:"""
    response = llm.invoke([prompt. format(context=context, query=query) ])
    
    output = {
        'answer': response. content,
        'sources': sources,
        'confidence': confidence
    }

    if return_context:
        output['context' ] = context
    return output



In [38]:
# Example usage:
result = rag_advanced("What is Encoder and Decoder Stacks?", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer' ])
print("\nSources:", result['sources' ])
print("\nConfidence:", result['confidence' ])
print("\nContext Preview:", result['context' ] [:300] )


Retrieving documents for query:  'What is Encoder and Decoder Stacks?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 86.69it/s]

Generated embeddings with shape: (1, 384)
Retrieved 2 documents (after filtering)


Answer: The Encoder and Decoder Stacks are the core components of the Transformer model, each composed of a stack of N = 6 identical layers.

*   **Encoder:** Each layer contains two sub-layers: a multi-head self-attention mechanism and a position-wise fully connected feed-forward network.
*   **Decoder:** Each layer contains three sub-layers: a masked multi-head self-attention mechanism, a multi-head attention mechanism over the encoder output, and a position-wise fully connected feed-forward network.

Both stacks employ residual connections and layer normalization around each sub-layer, with all sub-layers producing outputs of dimension $d_{model} = 512$.

Sources: [{'source': 'attention.pdf', 'page': 1, 'score': 0.22753143310546875, 'preview': 'Encoder: The encoder is composed of a stack of N = 6 identical layers. Each layer has two\nsub-layers. The ﬁrst is a mult...'}, {'source': 'attention.pdf', 'page': 1, 'score': 0.15884125232696533, 'preview': 'discuss its advantages over model